# LeetCode 2274 — Maximum Consecutive Floors Without Special Floors bằng Python

Link đề: https://leetcode.com/problems/maximum-consecutive-floors-without-special-floors/ · Độ khó: **Medium** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxConsecutive(self, bottom: int, top: int, special: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đưa bài "dãy tầng liên tiếp dài nhất không chứa tầng đặc biệt" về bài **khoảng trống lớn nhất giữa các điểm đã sắp xếp**.
2. Dùng **lính canh** (sentinel) `bottom − 1` và `top + 1` để xử lý hai khoảng ở **biên** cùng một công thức.
3. Hiểu vì sao không thể duyệt từng tầng (khoảng tới 10⁹) — phải làm việc trên **n điểm đặc biệt**.
4. Biết lời giải **O(n) không sắp xếp** bằng **xô (bucket) + nguyên lý chuồng bồ câu** (cùng ý tưởng với LC 164 *Maximum Gap*).
5. Kiểm chứng với vét cạn trên khoảng nhỏ.

## 2. Đề bài & ví dụ

Alice thuê mọi tầng từ `bottom` tới `top` (bao gồm hai đầu). Một số tầng là **tầng đặc biệt** (để nghỉ ngơi), cho trong mảng `special` (các giá trị phân biệt, đều trong `[bottom, top]`).

Trả về **số tầng liên tiếp nhiều nhất** không chứa tầng đặc biệt nào.

**Ví dụ 1:** `bottom = 2, top = 9, special = [4,6]` → `3`: các khoảng `2..3` (2 tầng), `5..5` (1 tầng), `7..9` (3 tầng).

**Ví dụ 2:** `bottom = 6, top = 8, special = [7,6,8]` → `0` (mọi tầng đều đặc biệt).

**Ràng buộc:** `1 ≤ len(special) ≤ 10⁵`, `1 ≤ bottom ≤ special[i] ≤ top ≤ 10⁹`, các giá trị phân biệt.

In [1]:
from typing import List
import random, time

examples = [(2, 9, [4, 6], 3), (6, 8, [7, 6, 8], 0)]
def draw(bottom, top, special):
    sp = set(special)
    print("tầng:", " ".join(f"{f:>2}" for f in range(bottom, top + 1)))
    print("     ", " ".join(" S" if f in sp else " ." for f in range(bottom, top + 1)))
for b, t, sp, exp in examples:
    draw(b, t, sp); print("kỳ vọng", exp)

tầng:  2  3  4  5  6  7  8  9
       .  .  S  .  S  .  .  .
kỳ vọng 3
tầng:  6  7  8
       S  S  S
kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Số phần tử nguyên giữa hai điểm (không tính hai đầu)

Giữa hai tầng đặc biệt `a < b` có `b − a − 1` tầng thường.

In [2]:
for a, b in [(4, 6), (4, 5), (1, 10)]:
    print(f"giữa {a} và {b}: {b - a - 1} tầng", list(range(a + 1, b)))

giữa 4 và 6: 1 tầng [5]
giữa 4 và 5: 0 tầng []
giữa 1 và 10: 8 tầng [2, 3, 4, 5, 6, 7, 8, 9]


### 3.2 Lính canh (sentinel)

Thêm "tầng đặc biệt ảo" `bottom − 1` và `top + 1`: khoảng đầu `bottom..s₀−1` trở thành "giữa `bottom − 1` và `s₀`", khoảng cuối tương tự ⇒ một công thức duy nhất cho mọi khoảng.

In [3]:
b, t, sp = 2, 9, [4, 6]
pts = [b - 1] + sorted(sp) + [t + 1]
print("điểm có lính canh:", pts, "-> các khoảng:", [y - x - 1 for x, y in zip(pts, pts[1:])])

điểm có lính canh: [1, 4, 6, 10] -> các khoảng: [2, 1, 3]


### 3.3 `zip(a, a[1:])` — duyệt các cặp liền kề

Sau khi sắp xếp, khoảng trống giữa các điểm liền kề là `a[k+1] − a[k] − 1`.

### 3.4 Nguyên lý chuồng bồ câu cho "khoảng cách lớn nhất"

Có `m` điểm trên đoạn dài `D`; khoảng cách lớn nhất giữa hai điểm liền kề **ít nhất** `D/(m − 1)`. Nếu chia đoạn thành các xô có độ rộng nhỏ hơn mức đó, khoảng lớn nhất **không thể nằm trong một xô** — nó nằm **giữa** các xô ⇒ chỉ cần min/max mỗi xô. Đây là nền tảng của lời giải O(n) (cách 5).

In [4]:
pts = [1, 3, 4, 10, 11]
D = pts[-1] - pts[0]; m = len(pts)
print("khoảng cách liền kề:", [y - x for x, y in zip(pts, pts[1:])], "| cận dưới D/(m-1) =", D / (m - 1))

khoảng cách liền kề: [2, 1, 6, 1] | cận dưới D/(m-1) = 2.5


## 4. Ý tưởng nền tảng / trực giác

Các tầng đặc biệt cắt đoạn `[bottom, top]` thành các **khoảng trống**. Đáp án là khoảng trống dài nhất. Có ba loại khoảng:
1. **Đầu:** từ `bottom` tới (tầng đặc biệt nhỏ nhất − 1): dài `s_min − bottom`.
2. **Giữa:** giữa hai tầng đặc biệt liền kề `a < b` (sau khi sắp xếp): dài `b − a − 1`.
3. **Cuối:** từ (tầng đặc biệt lớn nhất + 1) tới `top`: dài `top − s_max`.

Tính tay ví dụ 1: `sorted(special) = [4, 6]`:
- đầu: `4 − 2 = 2` (tầng 2, 3);
- giữa: `6 − 4 − 1 = 1` (tầng 5);
- cuối: `9 − 6 = 3` (tầng 7, 8, 9).

Max = 3. Ví dụ 2: `[6, 7, 8]`: đầu `6 − 6 = 0`, giữa `0, 0`, cuối `8 − 8 = 0` ⇒ 0.

Vì `top − bottom` tới 10⁹, **không** duyệt từng tầng; chỉ làm việc với `n ≤ 10⁵` điểm.

In [5]:
b, t, sp = 2, 9, sorted([4, 6])
print("đầu:", sp[0] - b, "| giữa:", [y - x - 1 for x, y in zip(sp, sp[1:])], "| cuối:", t - sp[-1])

đầu: 2 | giữa: [1] | cuối: 3


### 4.1 Hình dung các khoảng trống như các "thanh"

In mỗi khoảng trống thành một thanh dài bằng số tầng — đáp án là thanh dài nhất. Với nhiều tầng đặc biệt ngẫu nhiên, các thanh có độ dài rất khác nhau; tổng chiều dài các thanh luôn bằng số tầng thường.

In [6]:
rng0 = random.Random(5)
b0, t0 = 1, 60
sp0 = sorted(rng0.sample(range(b0, t0 + 1), 7))
pts0 = [b0 - 1] + sp0 + [t0 + 1]
for a, c in zip(pts0, pts0[1:]):
    g = c - a - 1
    print(f"{a + 1:>3}..{c - 1:<3} | {'#' * g} {g}")
print("tầng đặc biệt:", sp0, "| tổng thanh =", sum(c - a - 1 for a, c in zip(pts0, pts0[1:])), "= số tầng thường", (t0 - b0 + 1) - len(sp0))

  1..16  | ################ 16
 18..22  | ##### 5
 24..39  | ################ 16
 41..44  | #### 4
 46..47  | ## 2
 49..50  | ## 2
 52..53  | ## 2
 55..60  | ###### 6
tầng đặc biệt: [17, 23, 40, 45, 48, 51, 54] | tổng thanh = 53 = số tầng thường 53


## 5. Cách 1 — Brute Force / baseline: duyệt từng tầng

In [7]:
def brute(bottom, top, special):
    sp = set(special); best = cur = 0
    for f in range(bottom, top + 1):
        if f in sp: cur = 0
        else:
            cur += 1; best = max(best, cur)
    return best

for b, t, sp, exp in examples:
    got = brute(b, t, sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp:** O(top − bottom) — tới 10⁹ bước ⇒ TLE (bộ nhớ cho `set` thì ổn vì chỉ có n phần tử). Ý tưởng "đếm chuỗi dài nhất, reset khi gặp phần tử cấm" vẫn đáng nhớ cho bài có khoảng nhỏ.

**Nút thắt:** hầu hết các tầng là "tầng thường" nằm trong những khoảng dài; ta duyệt từng tầng thay vì nhảy thẳng từ tầng đặc biệt này sang tầng đặc biệt kế tiếp. Sắp xếp các tầng đặc biệt cho phép nhảy.

In [8]:
for span in [10**5, 10**6]:
    t0 = time.perf_counter(); brute(1, span, [span // 2]); print(f"khoảng {span:,} tầng: {(time.perf_counter()-t0)*1000:.0f} ms")

khoảng 100,000 tầng: 7 ms
khoảng 1,000,000 tầng: 55 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** sắp xếp `special` ⇒ các khoảng trống là `s₀ − bottom`, `s_{k+1} − s_k − 1`, `top − s_{n−1}`. Đáp án = max của `n + 1` số này. O(n log n).

**Với lính canh:** đặt `pts = [bottom − 1] + sorted(special) + [top + 1]` ⇒ mọi khoảng là `pts[k+1] − pts[k] − 1`.

**Quy tắc:**
- **Nếu** hai tầng đặc biệt kề nhau (`b = a + 1`) **thì** khoảng giữa dài 0 — không sao, công thức vẫn cho 0.
- **Nếu** tầng đặc biệt trùng `bottom` (hoặc `top`) **thì** khoảng đầu (cuối) dài 0.

**Tiến xa hơn — O(n) không sắp xếp:** khoảng trống lớn nhất giữa các điểm liền kề (kể cả lính canh) có thể tìm bằng **xô**: chia `[bottom − 1, top + 1]` thành `m − 1` xô (m = số điểm), mỗi xô chỉ nhớ min và max. Theo chuồng bồ câu, khoảng lớn nhất nằm giữa max của một xô và min của xô **không rỗng** kế tiếp. O(n) thời gian, O(n) bộ nhớ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + ba loại khoảng (cách doocs)

In [9]:
def sort_gaps(bottom, top, special):
    sp = sorted(special)
    best = max(sp[0] - bottom, top - sp[-1])
    for a, b in zip(sp, sp[1:]):
        best = max(best, b - a - 1)
    return best

for b, t, sp, exp in examples:
    got = sort_gaps(b, t, sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


**Độ phức tạp cách 2:** O(n log n) thời gian (sắp xếp), O(n) bộ nhớ cho bản sắp xếp (hoặc O(1) thêm nếu `special.sort()` tại chỗ). Không phụ thuộc vào độ dài đoạn `top − bottom`.

### 7.2 Cách 3 — Lính canh + một biểu thức (gọn nhất, ít lỗi biên nhất)

In [10]:
def sentinels(bottom, top, special):
    pts = [bottom - 1] + sorted(special) + [top + 1]
    return max(b - a - 1 for a, b in zip(pts, pts[1:]))

for b, t, sp, exp in examples:
    got = sentinels(b, t, sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


### 7.3 Cách 4 — Sắp xếp tại chỗ, duyệt chỉ số (O(1) bộ nhớ phụ)

In [11]:
def inplace(bottom, top, special):
    special = list(special)            # sao chép để demo không sửa input gốc; trên LeetCode có thể sort trực tiếp
    special.sort()
    best = special[0] - bottom
    for k in range(1, len(special)):
        d = special[k] - special[k - 1] - 1
        if d > best: best = d
    return max(best, top - special[-1])

for b, t, sp, exp in examples:
    print(inplace(b, t, sp), "OK" if inplace(b, t, sp) == exp else "SAI")

3 OK
0 OK


### 7.4 Cách 5 — Xô + chuồng bồ câu (O(n), không sắp xếp)

Các bước:
1. Điểm: `P = special ∪ {bottom − 1, top + 1}`, `m = |P|`, `lo = bottom − 1`, `hi = top + 1`.
2. Độ rộng xô `w = max(1, (hi − lo) // (m − 1))`; số xô `K = (hi − lo) // w + 1`.
3. Mỗi xô lưu `min`, `max` của các điểm rơi vào.
4. Duyệt xô theo thứ tự, khoảng cách ứng viên = `min(xô hiện tại) − max(xô không rỗng trước đó)`.
5. Đáp án = max ứng viên − 1 (số tầng **giữa**).

Vì `w ≤ (hi − lo)/(m − 1)` ≤ khoảng cách lớn nhất, hai điểm trong **cùng** một xô cách nhau tối đa `w − 1 < w` ⇒ không thể là khoảng lớn nhất nếu có khoảng ≥ w ở nơi khác — và theo chuồng bồ câu luôn có khoảng ≥ (hi − lo)/(m − 1) ≥ w. Do đó chỉ cần xét khoảng **giữa** các xô.

In [12]:
def bucket_max_gap(bottom, top, special):
    pts = special + [bottom - 1, top + 1]
    lo, hi = bottom - 1, top + 1; m = len(pts)
    w = max(1, (hi - lo) // (m - 1))
    K = (hi - lo) // w + 1
    bmin = [None] * K; bmax = [None] * K
    for p in pts:
        k = (p - lo) // w
        if bmin[k] is None or p < bmin[k]: bmin[k] = p
        if bmax[k] is None or p > bmax[k]: bmax[k] = p
    best = 0; prev_max = None
    for k in range(K):
        if bmin[k] is None: continue
        if prev_max is not None:
            best = max(best, bmin[k] - prev_max)
        prev_max = bmax[k]
    return best - 1

for b, t, sp, exp in examples:
    got = bucket_max_gap(b, t, sp)
    print(got, "OK" if got == exp else "SAI")

3 OK
0 OK


Lưu ý: khoảng lớn nhất **cũng có thể** nằm trong một xô khi mọi khoảng đều nhỏ hơn `w`? — không thể: tổng các khoảng liền kề bằng `hi − lo`, có `m − 1` khoảng ⇒ khoảng lớn nhất ≥ `(hi − lo)/(m − 1) ≥ w`. Hai điểm trong một xô cách nhau ≤ `w − 1`. Vậy khoảng lớn nhất luôn "vượt xô". (Mục 12 kiểm chứng ngẫu nhiên.)

**Độ phức tạp cách 5:** O(n) thời gian và bộ nhớ (K ≤ m xô).

## 8. TRACE từng bước

### 8.1 Trace cách 3 (lính canh)

In [13]:
def trace_sentinels(bottom, top, special):
    pts = [bottom - 1] + sorted(special) + [top + 1]; best = 0
    print("điểm:", pts)
    for a, b in zip(pts, pts[1:]):
        g = b - a - 1; best = max(best, g)
        print(f"  giữa {a:>3} và {b:>3}: {g} tầng {list(range(a + 1, b)) if g <= 6 else '...'} -> best = {best}")
    return best

trace_sentinels(2, 9, [4, 6]); print()
trace_sentinels(1, 20, [20, 1, 9, 10, 15])

điểm: [1, 4, 6, 10]
  giữa   1 và   4: 2 tầng [2, 3] -> best = 2
  giữa   4 và   6: 1 tầng [5] -> best = 2
  giữa   6 và  10: 3 tầng [7, 8, 9] -> best = 3

điểm: [0, 1, 9, 10, 15, 20, 21]
  giữa   0 và   1: 0 tầng [] -> best = 0
  giữa   1 và   9: 7 tầng ... -> best = 7
  giữa   9 và  10: 0 tầng [] -> best = 7
  giữa  10 và  15: 4 tầng [11, 12, 13, 14] -> best = 7
  giữa  15 và  20: 4 tầng [16, 17, 18, 19] -> best = 7
  giữa  20 và  21: 0 tầng [] -> best = 7


7

### 8.2 Trace cách 5 (xô)

In [14]:
def trace_bucket(bottom, top, special):
    pts = special + [bottom - 1, top + 1]; lo, hi = bottom - 1, top + 1; m = len(pts)
    w = max(1, (hi - lo) // (m - 1)); K = (hi - lo) // w + 1
    buckets = [[] for _ in range(K)]
    for p in pts: buckets[(p - lo) // w].append(p)
    print(f"lo={lo}, hi={hi}, m={m}, độ rộng xô w={w}, số xô K={K}")
    prev = None; best = 0
    for k, bk in enumerate(buckets):
        if not bk: print(f"  xô {k} [{lo + k*w}, {lo + (k+1)*w - 1}]: rỗng"); continue
        cand = min(bk) - prev if prev is not None else None
        if cand is not None: best = max(best, cand)
        print(f"  xô {k} [{lo + k*w}, {lo + (k+1)*w - 1}]: {sorted(bk)} -> khoảng với xô trước: {cand}")
        prev = max(bk)
    print("khoảng cách lớn nhất =", best, "-> số tầng =", best - 1)

trace_bucket(1, 20, [20, 1, 9, 10, 15])

lo=0, hi=21, m=7, độ rộng xô w=3, số xô K=8
  xô 0 [0, 2]: [0, 1] -> khoảng với xô trước: None
  xô 1 [3, 5]: rỗng
  xô 2 [6, 8]: rỗng
  xô 3 [9, 11]: [9, 10] -> khoảng với xô trước: 8
  xô 4 [12, 14]: rỗng
  xô 5 [15, 17]: [15] -> khoảng với xô trước: 5
  xô 6 [18, 20]: [20] -> khoảng với xô trước: 5
  xô 7 [21, 23]: [21] -> khoảng với xô trước: 1
khoảng cách lớn nhất = 8 -> số tầng = 7


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def maxConsecutive(self, bottom: int, top: int, special: List[int]) -> int:
        sp = sorted(special)
        best = max(sp[0] - bottom, top - sp[-1])     # khoảng ở hai biên
        for a, b in zip(sp, sp[1:]):                 # khoảng giữa hai tầng đặc biệt liền kề
            if b - a - 1 > best:
                best = b - a - 1
        return best

In [16]:
sol = Solution()
for b, t, sp, exp in examples:
    got = sol.maxConsecutive(b, t, sp)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

Gọi `s₀ < s₁ < … < s_{n−1}` là các tầng đặc biệt đã sắp xếp. Tập tầng thường `[bottom, top] ∖ {s_k}` là hợp **rời** của các đoạn liên tiếp:
`[bottom, s₀ − 1]`, `[s₀ + 1, s₁ − 1]`, …, `[s_{n−1} + 1, top]` (một số có thể rỗng).

Mọi dãy tầng liên tiếp không chứa tầng đặc biệt nằm **trọn** trong một đoạn như vậy (nếu vắt qua hai đoạn thì chứa tầng đặc biệt ở giữa). Ngược lại mỗi đoạn tự nó là một dãy hợp lệ. ⇒ đáp án = độ dài đoạn lớn nhất = `max(s₀ − bottom, s_{k+1} − s_k − 1, top − s_{n−1})`. Thuật toán tính đúng các số này. ∎

**Vì sao sắp xếp là "đủ"?** Sau khi sắp xếp, hai tầng đặc biệt **liền kề** trong danh sách là hai tầng không có tầng đặc biệt nào ở giữa — đúng là hai đầu mút của một đoạn trống. Không có cặp nào khác cần xét: một cặp không liền kề luôn có tầng đặc biệt ở giữa nên không tạo đoạn trống liên tục.

**Cận dưới về thời gian:** bài toán "khoảng cách lớn nhất giữa các phần tử liền kề sau khi sắp xếp" giải được trong O(n) bằng xô (cách 5) — nhanh hơn cận dưới O(n log n) của sắp xếp so sánh, vì ta không cần **thứ tự đầy đủ**, chỉ cần khoảng lớn nhất.

**Không âm:** vì `bottom ≤ s₀` và `s_{n−1} ≤ top`, các khoảng biên ≥ 0; khoảng giữa ≥ 0 vì các giá trị phân biệt.

## 11. Phân tích Time / Space Complexity

`n = len(special) ≤ 10⁵`, `D = top − bottom + 1 ≤ 10⁹`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng tầng | O(D) | O(n) |
| 2. Sắp xếp + 3 loại khoảng | O(n log n) | O(n) |
| 3. Lính canh | O(n log n) | O(n) |
| 4. Sắp xếp tại chỗ | O(n log n) | O(1) thêm |
| 5. Xô + chuồng bồ câu | O(n) | O(n) |

In [17]:
rng = random.Random(2274)
bottom, top = 1, 10**9
sp = rng.sample(range(bottom, top + 1), 10**5)
for name, f in [("sort_gaps", sort_gaps), ("sentinels", sentinels), ("inplace", inplace),
                ("bucket_max_gap", bucket_max_gap), ("Solution", Solution().maxConsecutive)]:
    t0 = time.perf_counter(); r = f(bottom, top, list(sp)); dt = time.perf_counter() - t0
    print(f"{name:<15} -> {r}  {dt*1000:.0f} ms")

sort_gaps       -> 128570  27 ms

sentinels       -> 128570  29 ms
inplace         -> 128570  22 ms
bucket_max_gap  -> 128570  26 ms


Solution        -> 128570  27 ms


(Trong Python, `sorted` viết bằng C nên cách 2/3 thường nhanh hơn cách xô dù độ phức tạp lý thuyết kém hơn — hằng số quan trọng!)

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "sort_gaps": sort_gaps, "sentinels": sentinels, "inplace": inplace,
       "bucket_max_gap": bucket_max_gap, "Solution": Solution().maxConsecutive}
edge = examples + [
    (1, 1, [1], 0),
    (1, 10, [1], 9),                       # chỉ khoảng cuối
    (1, 10, [10], 9),                      # chỉ khoảng đầu
    (1, 10, [5], 5),                       # đầu 4, cuối 5
    (1, 10, [3, 4, 5], 5),                 # đặc biệt liền nhau
    (5, 5, [5], 0),
    (1, 10**9, [1, 10**9], 10**9 - 2),
    (1, 10**9, [5 * 10**8], 5 * 10**8),
]
for b, t, sp, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and t - b > 10**6: continue
        a = list(sp)
        assert f(b, t, a) == exp, (name, b, t, sp, f(b, t, sp), exp)
        assert a == sp, (name, "đã sửa input")
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(3000):
    b = rng.randint(1, 50); t = b + rng.randint(0, 60)
    k = rng.randint(1, t - b + 1)
    sp = rng.sample(range(b, t + 1), k)
    ref = brute(b, t, sp)
    for name, f in ALL.items():
        assert f(b, t, list(sp)) == ref, (name, b, t, sp)
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Tính chất: tổng các khoảng = số tầng thường; đáp án ≥ ⌈(số tầng thường) / (n + 1)⌉

In [20]:
rng = random.Random(7)
for _ in range(500):
    b = rng.randint(1, 1000); t = b + rng.randint(0, 2000)
    sp = rng.sample(range(b, t + 1), rng.randint(1, min(50, t - b + 1)))
    pts = [b - 1] + sorted(sp) + [t + 1]
    gaps = [y - x - 1 for x, y in zip(pts, pts[1:])]
    normal = (t - b + 1) - len(sp)
    assert sum(gaps) == normal
    ans = Solution().maxConsecutive(b, t, sp)
    assert ans >= -(-normal // (len(sp) + 1))
print("tính chất OK")

tính chất OK


Cận dưới thứ hai chính là nguyên lý chuồng bồ câu: `normal` tầng chia vào `n + 1` khoảng ⇒ có khoảng ≥ trung bình.

### 12.4 Cực trị theo ràng buộc

- `special` gồm **mọi** tầng của một đoạn dài 10⁵ ⇒ 0.
- Một tầng đặc biệt ở giữa đoạn `[1, 10⁹]` ⇒ khoảng lớn nhất ≈ 5·10⁸.
- `special` gồm 10⁵ tầng cách đều trong `[1, 10⁹]` ⇒ khoảng ≈ 10⁴.

In [21]:
all_special = list(range(1, 10**5 + 1))
even_spread = list(range(1, 10**9 + 1, 10**4))
for name, f in ALL.items():
    if name == "brute": continue
    assert f(1, 10**5, list(all_special)) == 0
    assert f(1, 10**9, [5 * 10**8 + 1]) == 5 * 10**8
    assert f(1, 10**9, list(even_spread)) == 10**4 - 1
print("cực trị OK; cách đều ->", Solution().maxConsecutive(1, 10**9, even_spread))

cực trị OK; cách đều ->

 9999


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên hai khoảng ở biên

In [22]:
def wrong_no_edges(bottom, top, special):
    sp = sorted(special)
    return max((b - a - 1 for a, b in zip(sp, sp[1:])), default=0)       # SAI

print("bản sai :", wrong_no_edges(2, 9, [4, 6]), "| bản đúng:", Solution().maxConsecutive(2, 9, [4, 6]))

bản sai : 1 | bản đúng: 3


**Sửa:** thêm `sp[0] − bottom` và `top − sp[-1]` (hoặc dùng lính canh).

### Lỗi 2 — Không sắp xếp `special`

In [23]:
def wrong_no_sort(bottom, top, special):
    sp = special                                                          # SAI: input chưa sắp xếp
    best = max(sp[0] - bottom, top - sp[-1])
    for a, b in zip(sp, sp[1:]): best = max(best, b - a - 1)
    return best

print("bản sai :", wrong_no_sort(6, 8, [7, 6, 8]), "| bản đúng:", Solution().maxConsecutive(6, 8, [7, 6, 8]))
print("bản sai :", wrong_no_sort(1, 20, [15, 3]), "| bản đúng:", Solution().maxConsecutive(1, 20, [15, 3]))

bản sai : 1 | bản đúng: 0
bản sai : 17 | bản đúng: 11


**Vì sao sai:** ví dụ 2 cho `special = [7, 6, 8]` chưa sắp xếp; hiệu `6 − 7 − 1 = −2`, khoảng đầu `7 − 6 = 1` (sai, tầng 6 là đặc biệt). **Sửa:** `sorted(special)`.

### Lỗi 3 — Lệch một: `b − a` thay vì `b − a − 1`

In [24]:
def wrong_off_by_one(bottom, top, special):
    pts = [bottom - 1] + sorted(special) + [top + 1]
    return max(b - a for a, b in zip(pts, pts[1:]))                      # SAI

print("bản sai :", wrong_off_by_one(2, 9, [4, 6]), "| bản đúng:", Solution().maxConsecutive(2, 9, [4, 6]))

bản sai : 4 | bản đúng: 3


**Sửa:** số tầng **giữa** hai tầng `a < b` là `b − a − 1`.

### Lỗi 4 — Lính canh sai (`bottom` và `top` thay vì `bottom − 1` và `top + 1`)

In [25]:
def wrong_sentinels(bottom, top, special):
    pts = [bottom] + sorted(special) + [top]                              # SAI
    return max(b - a - 1 for a, b in zip(pts, pts[1:]))

print("bản sai :", wrong_sentinels(2, 9, [4, 6]), "| bản đúng:", Solution().maxConsecutive(2, 9, [4, 6]))
print("bản sai :", wrong_sentinels(6, 8, [7, 6, 8]), "| bản đúng:", Solution().maxConsecutive(6, 8, [7, 6, 8]))

bản sai : 2 | bản đúng: 3
bản sai : 0 | bản đúng: 0


**Vì sao sai:** lính canh `bottom` coi tầng `bottom` là đặc biệt ⇒ khoảng đầu bị hụt 1; ngoài ra nếu `bottom` cũng là tầng đặc biệt thì sinh hiệu `−1`. **Sửa:** `bottom − 1` và `top + 1` (nằm **ngoài** đoạn thuê).

### Lỗi 5 — Duyệt từng tầng với khoảng 10⁹ (TLE)

In [26]:
def brute_with_limit(bottom, top, special, cap=2 * 10**6):
    if top - bottom + 1 > cap:
        return f"quá chậm: cần duyệt {top - bottom + 1:,} tầng"
    return brute(bottom, top, special)

print(brute_with_limit(1, 10**9, [5]), "| bản đúng:", Solution().maxConsecutive(1, 10**9, [5]))

quá chậm: cần duyệt 1,000,000,000 tầng | bản đúng: 999999995


**Sửa:** chỉ làm việc trên `n` tầng đặc biệt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn liên tiếp dài nhất không chứa điểm bị cấm" trên trục dài ⇒ **sắp xếp điểm cấm + khoảng trống liền kề**.
- **Lính canh** ở hai đầu biến các trường hợp biên thành trường hợp thường.
- "Khoảng cách lớn nhất giữa các phần tử liền kề sau khi sắp xếp" trong O(n) ⇒ **xô + chuồng bồ câu** (LC 164).
- Tọa độ lớn (10⁹) ⇒ không bao giờ duyệt từng tọa độ.
- Trong Python, `sorted` (C) thường nhanh hơn thuật toán O(n) viết bằng vòng lặp Python — chọn cách đơn giản khi ràng buộc cho phép.
- Bài cùng họ: 164 Maximum Gap, 228 Summary Ranges, 163 Missing Ranges, 1637 Widest Vertical Area Between Two Points, 2943.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **đoạn** `[lo, hi]` dài nhất (nhỏ nhất nếu hòa), hoặc `None`

In [27]:
def longest_range(bottom, top, special):
    pts = [bottom - 1] + sorted(special) + [top + 1]; best = None
    for a, b in zip(pts, pts[1:]):
        if b - a - 1 > 0 and (best is None or b - a - 1 > best[1] - best[0] + 1):
            best = (a + 1, b - 1)
    return best

print(longest_range(2, 9, [4, 6]), longest_range(6, 8, [7, 6, 8]), longest_range(1, 10, [5]))

(7, 9) None (6, 10)


### 15.2 Biến thể B — Được **bỏ** tối đa `k` tầng đặc biệt (biến chúng thành tầng thường)

Đoạn tốt nhất trải qua `k` tầng đặc biệt liên tiếp (sau sắp xếp) ⇒ cửa sổ trượt trên mảng lính canh: chọn `pts[i]` và `pts[i + k + 1]` làm hai "vách", độ dài `pts[i+k+1] − pts[i] − 1`.

In [28]:
def max_consecutive_remove_k(bottom, top, special, k):
    pts = [bottom - 1] + sorted(special) + [top + 1]
    k = min(k, len(special))
    return max(pts[i + k + 1] - pts[i] - 1 for i in range(len(pts) - k - 1))

def remove_k_brute(bottom, top, special, k):
    from itertools import combinations
    best = 0
    for r in range(0, min(k, len(special)) + 1):
        for rem in combinations(special, r):
            rest = [x for x in special if x not in rem]
            best = max(best, brute(bottom, top, rest))
    return best

rng = random.Random(152)
for _ in range(300):
    b = rng.randint(1, 10); t = b + rng.randint(0, 20)
    sp = rng.sample(range(b, t + 1), rng.randint(1, min(6, t - b + 1))); k = rng.randint(0, 3)
    assert max_consecutive_remove_k(b, t, sp, k) == remove_k_brute(b, t, sp, k)
print(max_consecutive_remove_k(2, 9, [4, 6], 1), max_consecutive_remove_k(2, 9, [4, 6], 2), "- khớp brute force")

5 8 - khớp brute force


### 15.3 Biến thể C — Nhiều truy vấn `(bottom, top)` trên cùng tập `special`

Sắp xếp một lần; mỗi truy vấn: dùng `bisect` tìm các tầng đặc biệt trong `[bottom, top]`, khoảng biên tính trực tiếp, khoảng giữa là max trên một đoạn của mảng khoảng ⇒ có thể dùng **sparse table** để O(1) mỗi truy vấn. Bản đơn giản O(số điểm trong đoạn):

In [29]:
from bisect import bisect_left, bisect_right
def queries(special, qs):
    sp = sorted(special); out = []
    for b, t in qs:
        i, j = bisect_left(sp, b), bisect_right(sp, t)
        inside = sp[i:j]
        if not inside: out.append(t - b + 1); continue
        best = max(inside[0] - b, t - inside[-1])
        for x, y in zip(inside, inside[1:]): best = max(best, y - x - 1)
        out.append(best)
    return out

print(queries([4, 6, 15], [(2, 9), (5, 14), (16, 30), (1, 100)]))

[3, 8, 15, 85]


### 15.4 Biến thể D — Tầng đặc biệt cho dưới dạng **đoạn** `[a, b]` (có thể chồng nhau)

Gộp các đoạn chồng/kề nhau (merge intervals), rồi tính khoảng trống giữa các đoạn đã gộp, cộng hai biên.

In [30]:
def max_consecutive_ranges(bottom, top, ranges):
    rs = sorted(ranges); merged = []
    for a, b in rs:
        if merged and a <= merged[-1][1] + 1: merged[-1][1] = max(merged[-1][1], b)
        else: merged.append([a, b])
    best = max(merged[0][0] - bottom, top - merged[-1][1])
    for (a1, b1), (a2, b2) in zip(merged, merged[1:]):
        best = max(best, a2 - b1 - 1)
    return best

def ranges_brute(bottom, top, ranges):
    sp = sorted({x for a, b in ranges for x in range(a, b + 1)})
    return brute(bottom, top, sp)

rng = random.Random(154)
for _ in range(500):
    b = rng.randint(1, 10); t = b + rng.randint(0, 40)
    rs = []
    for _ in range(rng.randint(1, 5)):
        a = rng.randint(b, t); rs.append([a, min(t, a + rng.randint(0, 6))])
    assert max_consecutive_ranges(b, t, [list(x) for x in rs]) == ranges_brute(b, t, rs)
print(max_consecutive_ranges(1, 100, [[10, 20], [15, 30], [31, 31], [60, 70]]), "- khớp brute force")

30 - khớp brute force


Chú ý gộp cả các đoạn **kề nhau** (`a ≤ b_prev + 1`), nếu không sẽ sinh khoảng "0 tầng" giữa chúng — vô hại với `max` nhưng tốn công.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt tầng | đếm chuỗi tầng thường | O(D) | O(n) | kiểm chứng D nhỏ |
| 2. Sắp xếp + 3 loại | khoảng biên + giữa | O(n log n) | O(n) | **mặc định** |
| 3. Lính canh | một công thức | O(n log n) | O(n) | gọn, ít lỗi biên |
| 4. Tại chỗ | sort trực tiếp | O(n log n) | O(1) | bộ nhớ chặt |
| 5. Xô | chuồng bồ câu | O(n) | O(n) | cần O(n) lý thuyết |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxConsecutive(1, 100, [50])` và `maxConsecutive(3, 15, [7, 9, 13])`?

**Bài 2 (tính tay):** Với `bottom = 10`, `top = 40`, `special = [12, 25, 26, 33]`, liệt kê mọi khoảng trống và độ dài.

**Bài 3 (điền TODO):** Viết `bt3_todo(bottom, top, special)` trả về **số khoảng trống không rỗng**.

**Bài 4 (tự viết lại):** Viết `bt4_todo(bottom, top, special)` cài cách 5 (xô) không nhìn code.

**Bài 5 (mini challenge):** Được **thêm** tối đa `k` tầng đặc biệt mới (đặt ở đâu tùy ý trong `[bottom, top]`). Viết `bt5_todo(bottom, top, special, k)` trả về đáp án bài gốc **nhỏ nhất** có thể (chia nhỏ các khoảng lớn). Gợi ý: tìm nhị phân trên đáp án `X`: một khoảng dài `g` cần `⌈(g − X)/(X + 1)⌉` tầng mới để mọi phần ≤ X.

In [31]:
def bt3_todo(bottom, top, special):
    # TODO: số khoảng trống không rỗng
    pass

def bt4_todo(bottom, top, special):
    # TODO: xô + chuồng bồ câu
    pass

def bt5_todo(bottom, top, special, k):
    # TODO: thêm tối đa k tầng đặc biệt để đáp án nhỏ nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: khoảng đầu 49, cuối 50. Mức 2: lấy max. Mức 3: `50`; ví dụ hai: đầu `7 − 3 = 4`, giữa `1, 3`, cuối `15 − 13 = 2` ⇒ `4`.

**Bài 2.** Mức 1: lính canh 9 và 41. Mức 2: `10..11` (2), `13..24` (12), `26`/`25` kề nhau (0), `27..32` (6), `34..40` (7). Mức 3: đáp án 12.

**Bài 3.** Mức 1: lính canh. Mức 2: đếm các `b − a − 1 > 0`. Mức 3: `sum(1 for ...)`.

**Bài 4.** Mức 1: `w = max(1, (hi − lo) // (m − 1))`. Mức 2: mỗi xô min/max. Mức 3: max(min xô − max xô trước) − 1.

**Bài 5.** Mức 1: đáp án đơn điệu: X lớn thì cần ít tầng mới. Mức 2: với X cố định, khoảng `g > X` cần `⌈(g − X)/(X + 1)⌉` tầng (mỗi tầng mới "cắt" ra một phần dài X và chính nó). Mức 3: tìm nhị phân X ∈ [0, max gap], kiểm tra tổng ≤ k.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `50`, `4`. **Bài 2:** khoảng `2, 12, 0, 6, 7` ⇒ `12`.

In [32]:
assert Solution().maxConsecutive(1, 100, [50]) == 50
assert Solution().maxConsecutive(3, 15, [7, 9, 13]) == 4
pts = [9] + [12, 25, 26, 33] + [41]
assert [b - a - 1 for a, b in zip(pts, pts[1:])] == [2, 12, 0, 6, 7]

def bt3(bottom, top, special):
    pts = [bottom - 1] + sorted(special) + [top + 1]
    return sum(1 for a, b in zip(pts, pts[1:]) if b - a - 1 > 0)

def bt4(bottom, top, special):
    return bucket_max_gap(bottom, top, list(special))

def bt5(bottom, top, special, k):
    pts = [bottom - 1] + sorted(special) + [top + 1]
    gaps = [b - a - 1 for a, b in zip(pts, pts[1:])]
    def need(X):
        return sum(-(-(g - X) // (X + 1)) for g in gaps if g > X)
    lo, hi = 0, max(gaps)
    while lo < hi:
        mid = (lo + hi) // 2
        if need(mid) <= k: hi = mid
        else: lo = mid + 1
    return lo

def bt5_brute(bottom, top, special, k):
    from itertools import combinations
    free = [f for f in range(bottom, top + 1) if f not in set(special)]
    best = brute(bottom, top, special)
    for r in range(1, min(k, len(free)) + 1):
        for add in combinations(free, r):
            best = min(best, brute(bottom, top, special + list(add)))
    return best

rng = random.Random(1974)
for _ in range(400):
    b = rng.randint(1, 10); t = b + rng.randint(0, 12)
    sp = rng.sample(range(b, t + 1), rng.randint(1, min(4, t - b + 1)))
    assert bt3(b, t, sp) == sum(1 for g in [y - x - 1 for x, y in zip([b - 1] + sorted(sp), sorted(sp) + [t + 1])] if g > 0)
    assert bt4(b, t, sp) == brute(b, t, sp)
    k = rng.randint(0, 2)
    assert bt5(b, t, sp, k) == bt5_brute(b, t, sp, k), (b, t, sp, k)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: một khoảng dài `g` được chia bởi `c` tầng mới thành `c + 1` phần với tổng độ dài `g − c`; để mọi phần ≤ X cần `g − c ≤ X·(c + 1)` ⇔ `c ≥ (g − X)/(X + 1)`. Vì các khoảng độc lập, tổng số tầng mới tối thiểu là tổng các trần ⇒ kiểm tra được trong O(n), tìm nhị phân trên X.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pts = [bottom - 1] + sorted(special) + [top + 1]     # lính canh NGOÀI đoạn
return max(b - a - 1 for a, b in zip(pts, pts[1:]))
```

**Checklist:**
- [ ] **Sắp xếp** `special` (input có thể chưa sắp).
- [ ] Hai khoảng biên (hoặc lính canh `bottom − 1`, `top + 1`).
- [ ] `b − a − 1` (không tính hai tầng đặc biệt).
- [ ] Không duyệt từng tầng (10⁹).
- [ ] Đã thử: mọi tầng đặc biệt, đặc biệt ở hai đầu, đặc biệt liền nhau.
- [ ] Không sửa `special` của người gọi nếu không được phép (dùng `sorted`).